In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm

import pandas as pd


class Dataset:
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels
        assert len(labels) == len(features)

    def __len__(self):
        return len(self.labels)

    def num_classes(self):
        return len(set(self.labels))
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


class LinearProbe(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(LinearProbe, self).__init__()
        self.linear = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.linear(x)


def validate(model, loader, device='cuda'):
    model.eval()  # Set model to evaluation mode
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad():  # Disable gradient calculation for validation
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            outputs = model(x)

            _, predicted = torch.max(outputs, 1)
            correct_predictions += (predicted == y).sum().item()
            total_samples += y.size(0)

    accuracy = correct_predictions / total_samples * 100
    return accuracy


def predict(model, loader, device='cuda'):
    model.eval()  # Set model to evaluation mode
    outputs = []
    with torch.no_grad():  # Disable gradient calculation for validation
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            outputs.append(model(x).cpu())
    return torch.cat(outputs)



dataset_dir = '/FungiTastic'

features_train = torch.load('features/dinov2/full-500p-train.pth', weights_only=False)
train_df = pd.read_csv(f'{dataset_dir}/dataset/FungiTastic/metadata/FungiTastic/FungiTastic-Train.csv')
train_df = train_df.set_index('filename').loc[features_train['files']].reset_index()
labels_train = train_df['category_id'].values


features_val = torch.load('features/dinov2/full-500p-val.pth', weights_only=False)
val_df = pd.read_csv(f'{dataset_dir}/dataset/FungiTastic/metadata/FungiTastic/FungiTastic-OpenSet-Val.csv')
val_df = val_df.set_index('filename').loc[features_val['files']].reset_index()
labels_val = val_df['category_id'].values


features_test = torch.load('features/dinov2/full-500p-test.pth', weights_only=False)
test_df = pd.read_csv(f'{dataset_dir}/dataset/FungiTastic/metadata/FungiTastic/FungiTastic-test-DEV.csv')
test_df = test_df.set_index('filename').loc[features_test['files']].reset_index()
labels_test = test_df['category_id'].values


# DINOv2 logits
- Train simple linear classifier using DINOv2 features and obtain its logits.

In [ ]:
torch.manual_seed(0)

dataset_test = Dataset(features_test['features'], labels_test)
dataset_val = Dataset(features_val['features'], labels_val)
dataset_train = Dataset(features_train['features'], labels_train)


# Define hyperparameters
input_dim = 1024        # Input feature dimension
batch_size = 64         # Batch size for training
learning_rate = 0.0001   # Learning rate for the optimizer
num_epochs = 30         # Number of epochs to train


# Initialize the model, loss function, and optimizer
model = LinearProbe(input_dim=input_dim, num_classes=dataset_train.num_classes())
model.to('cuda')
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)


# Create DataLoader for the dataset
loader_train = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
loader_val = DataLoader(dataset_val, batch_size=batch_size, shuffle=False)
loader_test = DataLoader(dataset_test, batch_size=batch_size, shuffle=False)


# Training loop
for epoch in tqdm(range(num_epochs)):
    total_loss = 0
    correct_predictions = 0
    total_samples = 0

    for x, y in loader_train:
        x = x.to('cuda')
        y = y.to('cuda')
        # Forward pass
        outputs = model(x)
        loss = criterion(outputs, y)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Track metrics
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct_predictions += (predicted == y).sum().item()
        total_samples += y.size(0)
    
    # Print epoch statistics
    accuracy = correct_predictions / total_samples * 100
    val_acc = validate(model, loader_val)
    print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {total_loss/len(loader_train):.4f}, Train Accuracy: {accuracy:.2f}%,  Val Accuracy: {val_acc:.2f}%")



# Save the outputs
logits_train = predict(model, loader_train)
torch.save(
    {'logits': logits_train, 'files': features_train['files']},
    '/home/cermavo3/projects/fungi/features/dinov2/full-500p-logits-train.pth'
)

logits_test = predict(model, loader_test)
torch.save(
    {'logits': logits_test, 'files': features_test['files']},
    '/home/cermavo3/projects/fungi/features/dinov2/full-500p-logits-test.pth'
)
logits_val = predict(model, loader_val)
torch.save(
    {'logits': logits_val, 'files': features_val['files']},
    '/home/cermavo3/projects/fungi/features/dinov2/full-500p-logits-val.pth'
)


  3%|▎         | 1/30 [00:07<03:39,  7.56s/it]

Epoch [1/30], Train Loss: 3.6071, Train Accuracy: 36.35%,  Val Accuracy: 47.89%


  7%|▋         | 2/30 [00:14<03:26,  7.37s/it]

Epoch [2/30], Train Loss: 2.2609, Train Accuracy: 55.53%,  Val Accuracy: 55.23%


 10%|█         | 3/30 [00:21<03:14,  7.19s/it]

...

Epoch [28/30], Train Loss: 0.4899, Train Accuracy: 89.52%,  Val Accuracy: 67.52%


 97%|█████████▋| 29/30 [03:26<00:07,  7.14s/it]

Epoch [29/30], Train Loss: 0.4772, Train Accuracy: 89.81%,  Val Accuracy: 67.41%


100%|██████████| 30/30 [03:33<00:00,  7.13s/it]

Epoch [30/30], Train Loss: 0.4648, Train Accuracy: 90.11%,  Val Accuracy: 67.45%
